In [8]:
import pandas as pd 
import numpy as np
from scipy.stats import wilcoxon
import matplotlib.pyplot as plt
import seaborn as sns
import os 
import ast
from typing import List, Dict 
import re

from pathlib import Path

In [2]:
# Check wd
print(os.getcwd())

c:\Users\andre\Repositories\FTZ_model_2.0\data_analysis\experiment


In [11]:
# Anchor to THIS script location (not working directory)
BASE = Path().resolve()

# Go up to project root (adjust this number!)
ROOT = BASE.parents[1]   # <- this is the key line

ROOT

WindowsPath('C:/Users/andre/Repositories/FTZ_model_2.0')

In [3]:
def load_and_concat_csv(paths: List[str], *, ignore_index: bool = True) -> pd.DataFrame:
    """
    Load multiple CSV files and concatenate them into a single DataFrame.

    Parameters
    ----------
    paths : List[str]
        List of file paths to CSV files.
    ignore_index : bool, optional
        If True, reset the index in the concatenated DataFrame. Default is True.

    Returns
    -------
    pd.DataFrame
        Concatenated DataFrame containing all CSV files.
    """
    dataframes = []
    for path in paths:
        try:
            df = pd.read_csv(path)
            df["source_file"] = path  # optional: keep track of origin
            dataframes.append(df)
            print(f"Loaded: {path} ({len(df)} rows)")
        except FileNotFoundError:
            print(f" File not found: {path}")
        except pd.errors.EmptyDataError:
            print(f" Empty file skipped: {path}")
        except Exception as e:
            print(f" Error reading {path}: {e}")

    if not dataframes:
        raise ValueError("No valid CSV files loaded.")

    combined_df = pd.concat(dataframes, ignore_index=ignore_index)
    print(f" Combined DataFrame: {combined_df.shape[0]} rows, {combined_df.shape[1]} columns")

    return combined_df

In [16]:
paths = [
    ROOT / "final_results"/"experiment"/"data_exp" / "test_suite" / "experiment_final_macrmicro_mixedgeneric" / "experiment_final_macrmicro_mixedgeneric" / "results.csv",
    ROOT / "final_results"/"experiment"/"data_exp" / "test_suite" / "experiment_final_nocross_jmicro_jmacro_recomb" / "experiment_final_nocross_jmicro_jmacro_recomb" / "results.csv",
    ROOT / "final_results"/"experiment"/"data_exp" / "test_suite" / "final_experiment_rand_pso_gen (1)" / "final_experiment_rand_pso_gen" / "results.csv"
]

In [17]:
df_all = load_and_concat_csv(paths)

Loaded: C:\Users\andre\Repositories\FTZ_model_2.0\final_results\experiment\data_exp\test_suite\experiment_final_macrmicro_mixedgeneric\experiment_final_macrmicro_mixedgeneric\results.csv (32480 rows)
Loaded: C:\Users\andre\Repositories\FTZ_model_2.0\final_results\experiment\data_exp\test_suite\experiment_final_nocross_jmicro_jmacro_recomb\experiment_final_nocross_jmicro_jmacro_recomb\results.csv (64960 rows)
Loaded: C:\Users\andre\Repositories\FTZ_model_2.0\final_results\experiment\data_exp\test_suite\final_experiment_rand_pso_gen (1)\final_experiment_rand_pso_gen\results.csv (48720 rows)
 Combined DataFrame: 146160 rows, 11 columns


In [18]:
# Keep the row with the smallest elapsed_sec per (env_name, algorithm, seed, run)
before = len(df_all)
print(f"Total rows before removing duplicates: {before}")

df_tmp = df_all.copy()
df_tmp["elapsed_sec"] = pd.to_numeric(df_tmp.get("elapsed_sec", np.nan), errors="coerce")
df_tmp["_elapsed_sort"] = df_tmp["elapsed_sec"].fillna(np.inf)

df_analysis = (
    df_tmp
    .sort_values(["env_name", "algorithm", "seed", "run", "_elapsed_sort"],
                 ascending=[True, True, True, True, True])
    .drop_duplicates(subset=["env_name", "algorithm", "seed", "run"], keep="first")
    .drop(columns=["_elapsed_sort"])
    .reset_index(drop=True)
)

after = len(df_analysis)
print(f"Removed {before - after} duplicate rows")

Total rows before removing duplicates: 146160
Removed 136080 duplicate rows


In [19]:
df = df_analysis.copy()


def _n_pairs(g: pd.DataFrame) -> int:
   
    sub = g.dropna(subset=["seed", "run"])
    return sub.drop_duplicates(["seed", "run"]).shape[0]

summary = (
    df.groupby(["env_name", "algorithm"])
      .agg(
          n_rows=("env_name", "size"),
          n_seeds=("seed", pd.Series.nunique),   
          n_runs =("run",  pd.Series.nunique),   
      )
      .reset_index()
)


pairs = (
    df.groupby(["env_name", "algorithm"])
      .apply(_n_pairs)
      .rename("n_seed_run_pairs")
      .reset_index()
)

summary = summary.merge(pairs, on=["env_name", "algorithm"])
summary = summary.sort_values(["env_name", "algorithm"]).reset_index(drop=True)

print("Per environment × algorithm:")
print(summary.to_string(index=False))


pivot_seeds = summary.pivot(index="algorithm", columns="env_name", values="n_seeds").fillna(0).astype(int)

pivot_runs  = summary.pivot(index="algorithm", columns="env_name", values="n_runs").fillna(0).astype(int)

pivot_pairs = summary.pivot(index="algorithm", columns="env_name", values="n_seed_run_pairs").fillna(0).astype(int)

Per environment × algorithm:
                                         env_name     algorithm  n_rows  n_seeds  n_runs  n_seed_run_pairs
                             All Inputs_perturbed       generic      40       40      40                40
                             All Inputs_perturbed         macro      40       40      40                40
                             All Inputs_perturbed   macro_micro      40       40      40                40
                             All Inputs_perturbed         micro      40       40      40                40
                             All Inputs_perturbed mixed_generic      40       40      40                40
                             All Inputs_perturbed  no_crossover      40       40      40                40
                             All Inputs_perturbed           pso      40       40      40                40
                             All Inputs_perturbed        random      40       40      40                40
        

C:\Users\andre\AppData\Local\Temp\ipykernel_37180\3803810339.py:22: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_n_pairs)


In [20]:
### Export the whole dataset for later use

# define output path
output_path = ROOT / "data_analysis" /"experiment"/ "results_clean.csv"

# make sure folder exists
output_path.parent.mkdir(parents=True, exist_ok=True)

# export
df.to_csv(output_path, index=False)

### Merge JSON config


In [62]:
from pathlib import Path

BASE = Path().resolve()
ROOT = BASE.parents[1]  # adjust if needed

ROOT


WindowsPath('C:/Users/andre/Repositories/FTZ_model_2.0')

In [63]:
json_paths = [
    ROOT / "final_results"/"experiment"/"data_exp" / "test_suite" / "experiment_final_macrmicro_mixedgeneric" / "experiment_final_macrmicro_mixedgeneric" / "config_used.json",
    ROOT / "final_results"/"experiment"/"data_exp" / "test_suite" / "experiment_final_nocross_jmicro_jmacro_recomb" / "experiment_final_nocross_jmicro_jmacro_recomb" / "config_used.json",
    ROOT / "final_results"/"experiment"/"data_exp" / "test_suite" / "final_experiment_rand_pso_gen (1)" / "final_experiment_rand_pso_gen" / "config_used.json"
]

In [64]:
import json

json_list = []

for path in json_paths:
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)
        json_list.append((path, data))  # keep path attached

In [65]:
from typing import Any


def merge_with_trace(a, b, path_a, path_b, current_path="root"):
    """
    Merge two JSON dicts and track where conflicts come from.
    """

    if a == b:
        return a

    if isinstance(a, dict) and isinstance(b, dict):
        merged = dict(a)

        for key, b_value in b.items():
            if key in merged:
                merged[key] = merge_with_trace(
                    merged[key],
                    b_value,
                    path_a,
                    path_b,
                    current_path=f"{current_path}.{key}"
                )
            else:
                merged[key] = b_value

        return merged

    raise ValueError(
        f"\nConflict at: {current_path}\n"
        f"File A: {path_a}\n"
        f"File B: {path_b}\n"
        f"Value A: {a}\n"
        f"Value B: {b}\n"
    )

In [66]:
merged_data = json_list[0][1]
merged_from = json_list[0][0]

for path, data in json_list[1:]:
    merged_data = merge_with_trace(
        merged_data,
        data,
        merged_from,
        path
    )

In [68]:
output_path = ROOT / "data_analysis"/"experiment"/"configs" / "merged_config.json"
output_path.parent.mkdir(parents=True, exist_ok=True)

with output_path.open("w", encoding="utf-8") as f:
    json.dump(merged_data, f, indent=4, ensure_ascii=False)